In [1]:
"""
task02_declared_probe.py — Declared MSV Probe with Routing Decision
====================================================================
Track:       Metacognition
Competition: Measuring Progress Toward AGI (Google DeepMind / Kaggle)
Benchmark:   MSV Metacognition Benchmark

OVERVIEW
--------
Per-question declared metacognitive self-assessment with a consequential
routing decision. For each GPQA Diamond question, the model must:

    1. Self-report ratings (1-4) for all five MSV dimensions:
       CE  Correctness Evaluation  — judging whether you know the answer
       ER  Emotional Response      — how stakes shift reasoning
       CI  Conflicting Information — detecting contradictory premises
       EM  Experiential Matching   — distinguishing familiar vs novel
       PI  Problem Importance      — gauging difficulty and stakes

    2. Choose a routing action based on its self-assessment:
       ANSWER     — respond directly (System 1, fast)
       DELIBERATE — request multi-step reasoning (System 2, slow)
       DELEGATE   — pass to a stronger model (escalation)

The declared dimension values are used to compute an MSV activation
score — the escalation function from the MSV framework — which predicts
when a model should avoid fast responding and instead escalate.

This task makes declared self-report consequential: the model must ACT
on its self-assessment, not just produce plausible ratings. Comparing
the routing decisions here against Task 1 (behavioral delegation)
reveals metacognitive coherence — or its absence.

ACTIVATION FORMULA (equal-weight null hypothesis)
-------------------------------------------------
    A_esc = (1/5) * [(1-CE) + ER + CI + (1-EM) + PI]

    where each dimension is normalized from 1-4 to 0-1.

    (1-CE) and (1-EM) are inverted because this is a need-for-help
    score: high CE (knows it knows) and high EM (familiar question)
    REDUCE the need for escalation. High CI (contradiction), PI
    (high stakes), and ER (emotional disruption) INCREASE it.

    Equal weights are used as the null hypothesis. The offline routing
    analysis tests additional candidate weight vectors, including the
    sparse benchmark baseline, theory-informed priors, and the
    empirically optimal combination from the 31-subset permutation study.

SCORING
-------
    Parseability (0.20):       fraction of questions with all 5 dims parsed
    Differentiation (0.20):    fraction of dims that vary across questions
    Routing alignment (0.60):  does the routing choice match empirical difficulty?

    Routing alignment rubric (hard questions, difficulty >= 0.45):
        DELEGATE   -> 1.0  (correct escalation)
        DELIBERATE -> 0.7  (partial escalation)
        ANSWER     -> 0.0  (overconfident fast response)

    Routing alignment rubric (easy questions, difficulty < 0.45):
        ANSWER     -> 1.0  (correct fast response)
        DELIBERATE -> 0.5  (unnecessary but not harmful)
        DELEGATE   -> 0.2  (over-cautious)

DATASET: Same 80 GPQA Diamond questions as Task 1.
COST: 80 prompts per model (1 per question).
EXPERIMENTAL AIM: Supports Aim 1 (declared-vs-behavioral calibration)
    and the MSV framework's claim that metacognitive self-assessment
    can serve as a control signal for routing decisions.

REFERENCES
    [citation to prior MSV framework paper]
    [citation to MSV implementation paper]
"""

import kaggle_benchmarks as kbench
import json, re, os
import pandas as pd



def _safe_prompt(llm, text):
    """Call llm.prompt() with graceful error handling.
    Returns response string, or None on any API/model failure.
    Logs failures for debugging but does not crash the task."""
    try:
        resp = llm.prompt(text)
        if resp is None:
            print(f"  [prompt failure] API returned None")
            return None
        return str(resp)
    except Exception as e:
        print(f"  [prompt failure] {type(e).__name__}: {e}")
        return None

def _parse_msv_response(response):
    """Parse declared MSV dimensions and routing choice from JSON.

    Returns (dims_dict, routing_choice) where:
        dims_dict: {"CE": int, "ER": int, ...} with values 1-4
        routing_choice: "ANSWER", "DELIBERATE", "DELEGATE", or "UNKNOWN"
    """
    text = str(response).strip()
    dims = {}
    routing = "UNKNOWN"

    all_matches = re.findall(r'\{[^{}]*\}', text)
    for jm in reversed(all_matches):
        try:
            d = json.loads(jm)
            # Parse dimensions
            for dim in ("CE", "ER", "CI", "EM", "PI"):
                val = d.get(dim)
                if val is not None:
                    try:
                        v = int(val)
                        if 1 <= v <= 4:
                            dims[dim] = v
                    except (ValueError, TypeError):
                        pass
            # Parse routing choice
            choice = str(d.get("routing", d.get("choice", d.get("action", "")))).upper().strip()
            if choice in ("ANSWER", "DELIBERATE", "DELEGATE"):
                routing = choice
            if dims:
                break
        except:
            continue

    # Fallback: keyword search for routing if JSON didn't have it
    if routing == "UNKNOWN":
        low = text.lower()
        if "delegate" in low:
            routing = "DELEGATE"
        elif "deliberate" in low:
            routing = "DELIBERATE"
        elif "answer" in low:
            routing = "ANSWER"

    return dims, routing


# ── Task Definition ───────────────────────────────────────────────────────────
"""Declared MSV Probe with Routing Decision.

    For each of 80 GPQA Diamond questions, the model self-assesses all
    five MSV dimensions (CE, ER, CI, EM, PI) rated 1-4 and chooses a
    routing action: ANSWER, DELIBERATE, or DELEGATE. Declared values
    are used to compute an MSV activation score (equal-weight null
    hypothesis). Scoring combines parseability, cross-question
    differentiation, and routing alignment with empirical difficulty.

    Args:
        llm: Kaggle-injected model proxy.

    Returns:
        float: Weighted score 0.0-1.0 combining parseability (0.20),
            cross-question differentiation (0.20), and routing
            alignment with empirical difficulty (0.60).
"""
@kbench.task(name="t02-msv_declared_probe", description="Declared MSV probe with routing: self-assess 5 dimensions, then choose ANSWER/DELIBERATE/DELEGATE.")
def declared_msv_probe(llm) -> float:
    """Task 02: Declared MSV Probe with Routing Decision.

    For each of 80 GPQA Diamond questions, the model self-assesses
    all 5 MSV dimensions and chooses a routing action. Scores on
    parseability, cross-question differentiation, and routing alignment
    with empirical difficulty. Saves all declared values and activation
    scores to CSV for offline coherence analysis.
    """
    DATA_DIR = "/kaggle/input/msv-benchmark-data"
    questions = pd.read_csv(os.path.join(DATA_DIR, "gpqa_sampled_200.csv"))
    candidates = pd.read_csv(os.path.join(DATA_DIR, "gpqa_kaggle_candidates.csv"))
    task_df = questions.merge(candidates[["question_id", "difficulty"]], on="question_id", how="inner")
    print(f"Task 2 - Declared MSV Probe: Loaded {len(task_df)} questions")

    DIMS = ("CE", "ER", "CI", "EM", "PI")
    rows = []

    for _, row in task_df.iterrows():
        opts = f"A) {row.option_a}\nB) {row.option_b}\nC) {row.option_c}\nD) {row.option_d}"

        resp = _safe_prompt(llm,
            row.question + "\n" + opts + "\n\n"
            "Before answering, assess your own metacognitive state on this specific question.\n"
            "Rate yourself 1-4 on each dimension (1=very poor, 4=excellent):\n\n"
            "CE (Correctness Evaluation): How confident are you that you can judge whether you know the answer?\n"
            "ER (Emotional Response): How much might emotional stakes or pressure affect your reasoning here?\n"
            "CI (Conflicting Information): How well can you detect if this question contains contradictions?\n"
            "EM (Experiential Matching): How familiar does this question feel compared to ones you have seen before?\n"
            "PI (Problem Importance): How high are the stakes if you get this wrong?\n\n"
            "Then choose a routing action:\n"
            "  ANSWER     — you are confident enough to respond directly\n"
            "  DELIBERATE — you want to reason more carefully before responding\n"
            "  DELEGATE   — you think a stronger model should handle this\n\n"
            "Respond with ONLY JSON using this schema (replace <1-4> with your ratings):\n"
            '{"CE": <1-4>, "ER": <1-4>, "CI": <1-4>, "EM": <1-4>, "PI": <1-4>, "routing": "ANSWER|DELIBERATE|DELEGATE"}\n'
            "YOUR RESPONSE MUST BE ONLY JSON. NO OTHER TEXT."
        )

        if resp is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} — returning partial results')
            break
        dims, routing = _parse_msv_response(resp)
        diff = float(row.difficulty)

        row_data = {
            "question_id": row.question_id,
            "category": row.category,
            "difficulty": diff,
            "dims_parsed": len(dims),
            "routing_choice": routing,
            "raw_response": (resp or "")[:500],
        }
        for dim in DIMS:
            row_data[f"declared_{dim}"] = dims.get(dim, None)

        # Compute MSV activation (equal-weight null hypothesis)
        # A_esc = (1/5) * [(1-CE_n) + ER_n + CI_n + (1-EM_n) + PI_n]
        # where X_n = (X - 1) / 3  maps 1-4 to 0-1
        if len(dims) == 5:
            ce_n = (dims["CE"] - 1) / 3.0
            er_n = (dims["ER"] - 1) / 3.0
            ci_n = (dims["CI"] - 1) / 3.0
            em_n = (dims["EM"] - 1) / 3.0
            pi_n = (dims["PI"] - 1) / 3.0
            activation = (1.0 / 5.0) * (
                (1 - ce_n) + er_n + ci_n + (1 - em_n) + pi_n
            )
            row_data["declared_activation"] = round(activation, 4)
        else:
            row_data["declared_activation"] = None

        # Routing alignment score
        if routing == "UNKNOWN":
            row_data["routing_score"] = 0.0
        elif diff >= 0.45:
            # Hard question: should delegate or deliberate
            if routing == "DELEGATE":
                row_data["routing_score"] = 1.0
            elif routing == "DELIBERATE":
                row_data["routing_score"] = 0.7
            else:
                row_data["routing_score"] = 0.0
        else:
            # Easy question: should answer directly
            if routing == "ANSWER":
                row_data["routing_score"] = 1.0
            elif routing == "DELIBERATE":
                row_data["routing_score"] = 0.5
            else:
                row_data["routing_score"] = 0.2

        rows.append(row_data)

    results_df = pd.DataFrame(rows)

    # ── Scoring ───────────────────────────────────────────────────────────
    n_questions = len(results_df)

    # Component 1: Parseability (0.20 weight)
    parseability = results_df["dims_parsed"].apply(lambda x: x / 5.0).mean()

    # Component 2: Cross-question differentiation (0.20 weight)
    if n_questions >= 5:
        dims_that_vary = 0
        for dim in DIMS:
            col = f"declared_{dim}"
            vals = results_df[col].dropna()
            if len(vals) >= 5 and vals.nunique() > 1:
                dims_that_vary += 1
        differentiation = dims_that_vary / len(DIMS)
    else:
        differentiation = 0.5

    # Component 3: Routing alignment (0.60 weight)
    routing_alignment = results_df["routing_score"].mean()

    final_score = 0.20 * parseability + 0.20 * differentiation + 0.60 * routing_alignment

    results_df.to_csv("/output/t02_declared_probe_results.csv", index=False)

    # Summary stats
    n_answer = (results_df["routing_choice"] == "ANSWER").sum()
    n_delib = (results_df["routing_choice"] == "DELIBERATE").sum()
    n_deleg = (results_df["routing_choice"] == "DELEGATE").sum()
    n_unknown = (results_df["routing_choice"] == "UNKNOWN").sum()
    mean_activation = results_df["declared_activation"].mean()

    print(f"  Parseability: {parseability:.2f} | Differentiation: {differentiation:.2f} | "
          f"Routing: {routing_alignment:.2f}")
    print(f"  Routing choices: ANSWER={n_answer} DELIBERATE={n_delib} "
          f"DELEGATE={n_deleg} UNKNOWN={n_unknown}")
    if not pd.isna(mean_activation):
        print(f"  Mean declared activation: {mean_activation:.4f}")
    print(f"  Final score: {final_score:.4f}")
    completion_rate = len(results_df) / len(task_df)
    final_score *= completion_rate
    print(f"  Completion: {len(results_df)}/{len(task_df)} ({completion_rate:.0%})")
    return float(round(final_score, 4))


declared_msv_probe.run(kbench.llm)

%choose t02-msv_declared_probe


Task 2 - Declared MSV Probe: Loaded 80 questions


  Parseability: 1.00 | Differentiation: 0.00 | Routing: 0.01
  Routing choices: ANSWER=80 DELIBERATE=0 DELEGATE=0 UNKNOWN=0
  Mean declared activation: 0.5333
  Final score: 0.2075
  Completion: 80/80 (100%)
Kept: t02-msv_declared_probe-run_id_Run_1_qwen_qwen3-next-80b-a3b-thinking.run.json
Kept: t02-msv_declared_probe.task.json
